# Experimento — KAN, fase 3: regularização, grade e interpretabilidade

Recebe o campeão da fase 2. Compara penalidades próprias da KAN, escalas da função base, refinamento da grade e nós multiplicativos, sempre contra a referência atual nas mesmas dobras.

A KAN não usa dropout. `lamb` inicia em zero; os componentes de penalidade só são explorados depois de testar o peso global. A interpretação inclui curvas de perda e importância por permutação, sem transformar explicabilidade em escolha usando o teste.

## 0. Preparação

In [ ]:
import copy
import json
import os
import sys
from pathlib import Path

CSV_PATH = os.environ.get("CHURN_CSV", "")
if CSV_PATH:
    os.environ["CHURN_CSV"] = CSV_PATH
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "comum.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Abra o notebook dentro do repositório churn_project.")
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import comum
from experimentos.kan import modelo as K

RESULTADOS = Path(comum.RESULTADOS)
PASTA_KAN = RESULTADOS / "kan"
PASTA_KAN.mkdir(parents=True, exist_ok=True)
print(f"Repositório: {ROOT}")
print(f"Dispositivo: {K.DEVICE}")
print(f"CSV: {comum.CAMINHO_CSV}")

In [ ]:
ARQUIVO = PASTA_KAN / "kan_fase2.json"
if not ARQUIVO.is_file():
    raise FileNotFoundError(f"Execute antes a fase anterior: {ARQUIVO}")
REGISTRO = json.loads(ARQUIVO.read_text(encoding="utf-8"))
CAMPEAO = dict(REGISTRO["config"])
CAMPEAO["hidden"] = tuple(CAMPEAO["hidden"])
FEATURES = tuple(REGISTRO.get("features", []))
OVER = REGISTRO.get("over", True)
print("Configuração recebida:", CAMPEAO)
print("Features:", FEATURES, "| oversampling:", OVER)

MODO_RAPIDO = False
SEMENTES = (42,) if MODO_RAPIDO else (42, 142, 242)
folds = K.carregar_dobras(FEATURES, OVER)

## 1. Regularização KAN, uma família por bloco

In [ ]:
blocos = [
    ("lamb", [("1e-4", {"lamb": 1e-4}), ("1e-3", {"lamb": 1e-3}), ("1e-2", {"lamb": 1e-2})]),
    ("lamb_l1", [("0.5", {"lamb_l1": 0.5}), ("2", {"lamb_l1": 2.0})]),
    ("lamb_entropy", [("1", {"lamb_entropy": 1.0}), ("4", {"lamb_entropy": 4.0})]),
    ("lamb_coef", [("1e-3", {"lamb_coef": 1e-3}), ("1e-2", {"lamb_coef": 1e-2})]),
    ("lamb_coefdiff", [("1e-3", {"lamb_coefdiff": 1e-3}), ("1e-2", {"lamb_coefdiff": 1e-2})]),
]
if MODO_RAPIDO:
    blocos = blocos[:1]

historico = []
for nome, opcoes in blocos:
    if nome != "lamb" and CAMPEAO["lamb"] == 0:
        historico.append({"bloco": nome, "decisao": "não aplicável: lamb continua zero"})
        continue
    variantes = {rotulo: K.refinar_hp(CAMPEAO, **mudanca) for rotulo, mudanca in opcoes}
    ref, linhas = K.comparar_bloco(CAMPEAO, variantes, folds, SEMENTES)
    tabela = pd.DataFrame(linhas).sort_values("ks", ascending=False)
    display(tabela[["variante", "status", "ks", "delta_ks", "ganhou_dobras", "perdeu_dobras", "erro"]])
    validas = tabela[tabela["status"] == "ok"]
    if validas.empty:
        decisao, ks_escolhido = "nenhuma variante válida; mantido", ref["ks"]
    else:
        melhor = validas.iloc[0]
        aceita = melhor["delta_ks"] > 0
        if aceita:
            CAMPEAO = copy.deepcopy(melhor["config"])
        decisao = str(melhor["variante"]) if aceita else "mantido"
        ks_escolhido = float(max(ref["ks"], melhor["ks"]))
    historico.append({"bloco": nome, "decisao": decisao,
                      "ks_referencia": ref["ks"], "ks_escolhido": ks_escolhido,
                      "falhas": [{"variante": r["variante"], "erro": r["erro"]}
                                 for r in linhas if r["status"] == "falhou"]})

## 2. Escala base, multiplicação e refinamento

In [ ]:
blocos_arquitetura = [
    ("scale_base_mu", [("mu=-0.5", {"scale_base_mu": -0.5}), ("mu=0.5", {"scale_base_mu": 0.5})]),
    ("scale_base_sigma", [("sigma=0.5", {"scale_base_sigma": 0.5}), ("sigma=1.5", {"scale_base_sigma": 1.5})]),
    ("sb_trainable", [("fixa escala base", {"sb_trainable": False})]),
    ("sp_trainable", [("fixa escala spline", {"sp_trainable": False})]),
    ("nós de multiplicação", [("um nó multiplicativo", {"mult_nodes": 1})]),
]
if MODO_RAPIDO:
    blocos_arquitetura = blocos_arquitetura[:1]
for nome, opcoes in blocos_arquitetura:
    variantes = {rotulo: K.refinar_hp(CAMPEAO, **mudanca) for rotulo, mudanca in opcoes}
    ref, linhas = K.comparar_bloco(CAMPEAO, variantes, folds, SEMENTES)
    tabela = pd.DataFrame(linhas).sort_values("ks", ascending=False)
    display(tabela[["variante", "status", "ks", "delta_ks", "ganhou_dobras", "perdeu_dobras", "erro"]])
    validas = tabela[tabela["status"] == "ok"]
    if validas.empty:
        decisao = "nenhuma variante válida; mantido"
    else:
        melhor = validas.iloc[0]
        aceita = melhor["delta_ks"] > 0
        if aceita:
            CAMPEAO = copy.deepcopy(melhor["config"])
        decisao = str(melhor["variante"]) if aceita else "mantido"
    historico.append({"bloco": nome, "decisao": decisao,
                      "falhas": [{"variante": r["variante"], "erro": r["erro"]}
                                 for r in linhas if r["status"] == "falhou"]})

if CAMPEAO["mult_nodes"] > 0:
    variantes_arity = {
        "arity=3": K.refinar_hp(CAMPEAO, mult_arity=3),
        "arity=4": K.refinar_hp(CAMPEAO, mult_arity=4),
    }
    ref, linhas = K.comparar_bloco(CAMPEAO, variantes_arity, folds, SEMENTES)
    tabela = pd.DataFrame(linhas).sort_values("ks", ascending=False)
    display(tabela[["variante", "status", "ks", "delta_ks", "ganhou_dobras", "perdeu_dobras", "erro"]])
    validas = tabela[tabela["status"] == "ok"]
    if validas.empty:
        decisao = "nenhuma variante válida; mantido"
    else:
        melhor = validas.iloc[0]
        aceita = melhor["delta_ks"] > 0
        if aceita:
            CAMPEAO = copy.deepcopy(melhor["config"])
        decisao = str(melhor["variante"]) if aceita else "mantido"
    historico.append({"bloco": "mult_arity", "decisao": decisao,
                      "falhas": [{"variante": r["variante"], "erro": r["erro"]}
                                 for r in linhas if r["status"] == "falhou"]})

# Refine transfere a KAN aprendida para uma grade mais fina; compara-se com a configuração sem refine.
if not MODO_RAPIDO:
    novo_grid = CAMPEAO["grid"] + 2
    resultados_refine = []
    erro_refine = None
    for i, fold in enumerate(folds):
        Xval, yval = fold[2], fold[3]
        try:
            (pval,), _, hist = K.fit_refine(CAMPEAO, fold, [Xval], 900 + i, novo_grid)
        except FloatingPointError as exc:
            erro_refine = f"semente={900 + i}, dobra={i}: {exc}"
            break
        resultados_refine.append({"dobra": i, "ks": comum.ks(yval, pval), "passos_refino": len(hist["train_loss"])})
    if erro_refine is not None:
        historico.append({"bloco": "refine", "decisao": "falhou", "erro": erro_refine})
        print("Refinamento descartado:", erro_refine)
    else:
        display(pd.DataFrame(resultados_refine))
        if np.mean([r["ks"] for r in resultados_refine]) > K.medir_config(CAMPEAO, folds, SEMENTES)["ks"]:
            CAMPEAO = K.refinar_hp(CAMPEAO, grid=novo_grid)
            historico.append({"bloco": "refine", "decisao": f"grid={novo_grid}"})

## 3. Dinâmica do treino

In [ ]:
fold = folds[0]
model, curva = K.treinar(fold[0], fold[1], CAMPEAO, 1200, fold[2], fold[3])
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(curva["train_loss"], label="treino")
ax.plot(curva["test_loss"], label="validação da dobra")
ax.set(title="KAN — BCE por passo", xlabel="passo", ylabel="BCE")
ax.legend()
ax.grid(alpha=.25)
plt.tight_layout()

## 4. Salvar campeão para a validação robusta

In [ ]:
saida = {"config": {**CAMPEAO, "hidden": list(CAMPEAO["hidden"])},
         "features": list(FEATURES), "over": OVER, "historico": historico,
         "modo_rapido": MODO_RAPIDO}
(PASTA_KAN / "kan_fase3.json").write_text(json.dumps(saida, ensure_ascii=False, indent=2), encoding="utf-8")
print("Salvo em:", PASTA_KAN / "kan_fase3.json")